<a href="https://colab.research.google.com/github/Samlat-stack/titanic_survival_prediction/blob/main/titanic_survival_prediction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
import seaborn as sns

df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [79]:
df.shape
#no of rows and columns

(891, 15)

In [80]:
df["survived"].value_counts()


,count
survived,
0,549
1,342


In [81]:
df.groupby("sex")["survived"].mean()


,survived
sex,
female,0.742038
male,0.188908


In [82]:
df.isna().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [83]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Pick useful columns (leaving out 'deck')
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()

# 2. Fix missing ages and turn text into numbers
data["age"] = data["age"].fillna(data["age"].median())
data["sex"] = data["sex"].map({"male": 0, "female": 1})

# 3. Split into inputs (X) and the answer (y)
X = data.drop("survived", axis=1)
y = data["survived"]

# 4. Hold back 20% of the data to test the model on
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Train and score
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.7988826815642458


# Titanic Survival Prediction

## Goal
Predict whether a passenger survived the Titanic using basic information
such as ticket class, sex, age, fare, and family size.

## Data and cleaning
- Dataset: the Titanic dataset (loaded from the seaborn library).
- The `deck` column was dropped because about 77% of its values were missing.
- Missing ages were filled with the median age.
- `sex` was converted to numbers (male = 0, female = 1) so the models could use it.
- Data was split 80% for training and 20% for testing.

## Key observation
Survival differed sharply by sex: about 74% of women survived compared with
about 19% of men.

## Models and results
I compared a random forest and a logistic regression, first using
pclass, sex, age, and fare, then adding sibsp and parch (family size).

| Model               | Accuracy (4 features) | Accuracy (6 features) |
|---------------------|-----------------------|-----------------------|
| Random forest       | 0.793                 | 0.799                 |
| Logistic regression | 0.800                 | 0.810                 |

In the random forest, `fare` had the highest feature importance.

## What I learned
- The simpler model (logistic regression) performed as well as or slightly
  better than the random forest.
- The differences between scores are small (about one or two passengers in
  the test set), so I can't be sure from a single split that adding family
  columns truly helped.


## Next steps
- Use cross-validation for a more reliable score.
- Add more features such as `embarked` and a family-size feature.
- Try other models.

In [84]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

fare      0.304682
sex       0.274729
age       0.253842
pclass    0.082890
sibsp     0.048588
parch     0.035268
dtype: float64


In [85]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
print("Logistic Regression accuracy:", lr.score(X_test, y_test))

Logistic Regression accuracy: 0.8100558659217877


In [86]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier

models = {
    "Decision tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Gradient boosting": GradientBoostingClassifier(random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, m.score(X_test, y_test))

Decision tree 0.7541899441340782
KNN 0.6871508379888268
Gradient boosting 0.8044692737430168


In [87]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(random_state=42),
    "Gradient boosting": GradientBoostingClassifier(random_state=42),
}

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5)
    print(name, "average:", round(scores.mean(), 3), "| spread:", round(scores.std(), 3))

Logistic regression average: 0.785 | spread: 0.02
Random forest average: 0.81 | spread: 0.033
Gradient boosting average: 0.834 | spread: 0.013


## Cross-validation (5-fold)
A single train/test split gave misleading results, so I compared the
models with 5-fold cross-validation:

| Model               | Mean accuracy | Std dev |
|---------------------|---------------|---------|
| Gradient boosting   | 0.834         | 0.013   |
| Random forest       | 0.810         | 0.033   |
| Logistic regression | 0.785         | 0.020   |

Gradient boosting performed best and most consistently. Logistic regression
looked best on a single split but ranked last under cross-validation,
which shows how much a single split can mislead.
